# Module 08 — GPU Hardware Selection

This notebook compares representative T4, A100, H100, H200, and B200 constraints, estimates model fit, calculates memory-streaming lower bounds, and optionally inspects the attached Lightning GPU.

The central rule is: **capacity is a gate; bandwidth, compute, interconnect, and cost determine speed after the workload fits.**

## Learning goals

- compare GPU capacity, memory bandwidth, precision, and interconnect constraints;
- estimate whether weights, KV cache, and runtime headroom fit;
- compare idealized memory-streaming lower bounds;
- inspect the actual GPU and topology available to the notebook.

In [ ]:
# Standalone planning model: this notebook does not import hardware_selection.py.
from dataclasses import dataclass
import subprocess

@dataclass(frozen=True)
class GpuSpec:
    name: str; architecture: str; memory_type: str; memory_gb: float; bandwidth_gbs: float; precision: str; interconnect: str
    @property
    def usable_gb(self): return self.memory_gb * 0.90

GPU_SPECS = (
    GpuSpec('T4', 'Turing', 'GDDR6', 16, 300, 'FP16 · INT8 · INT4', 'PCIe Gen3'),
    GpuSpec('A100 80GB', 'Ampere', 'HBM2e', 80, 2040, 'FP16 · BF16 · TF32', 'PCIe/NVLink'),
    GpuSpec('H100', 'Hopper', 'HBM3', 80, 3350, 'FP16 · BF16 · FP8', 'PCIe/NVLink'),
    GpuSpec('H200', 'Hopper', 'HBM3e', 141, 4800, 'FP16 · BF16 · FP8', 'PCIe/NVLink'),
    GpuSpec('B200', 'Blackwell', 'HBM3e', 180, 8000, 'FP16 · BF16 · FP8 · FP4', 'NVLink/NVSwitch'),
)

def print_specs():
    for spec in GPU_SPECS: print(spec)

def estimate_budget(parameters_b, weight_bytes, reserve_gb, kv_gb_per_1k_tokens, tokens):
    weights_gb = parameters_b * weight_bytes; kv_gb = kv_gb_per_1k_tokens * tokens / 1000
    return {'weights_gb': weights_gb, 'kv_gb': kv_gb, 'reserve_gb': reserve_gb, 'total_gb': weights_gb + kv_gb + reserve_gb}

def print_budget(parameters_b, weight_bytes, reserve_gb, kv_gb_per_1k_tokens, tokens):
    budget = estimate_budget(parameters_b, weight_bytes, reserve_gb, kv_gb_per_1k_tokens, tokens)
    print('Budget:', budget)
    for spec in GPU_SPECS: print(spec.name, 'fits:', budget['total_gb'] <= spec.usable_gb, 'headroom:', spec.usable_gb - budget['total_gb'])

def print_lower_bounds(weight_gb):
    for spec in GPU_SPECS: print(spec.name, weight_gb / spec.bandwidth_gbs * 1000, 'ms ideal read bound')

def probe_device(show_topology=False):
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('CUDA is unavailable')
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index); free, total = torch.cuda.mem_get_info(index)
        print(index, props.name, f'{total / 2**30:.2f} GiB total', f'{free / 2**30:.2f} GiB free')
    if show_topology: subprocess.run(['nvidia-smi', 'topo', '-m'], check=False)

print('Standalone GPU planning model loaded')

## 1. Compare representative GPU constraints

These are published-style reference values for specific variants, not application benchmarks. Check the exact SKU before deployment.

In [ ]:
# Experiment 1: print the representative capacity, bandwidth, and link table.
for spec in GPU_SPECS:
    print(f'{spec.name:10s} | {spec.architecture:9s} | {spec.memory_gb:5.0f} GB | {spec.bandwidth_gbs:5.0f} GB/s | {spec.precision} | {spec.interconnect}')

In [ ]:
import matplotlib.pyplot as plt

names = [spec.name for spec in GPU_SPECS]
memory = [spec.memory_gb for spec in GPU_SPECS]
bandwidth = [spec.bandwidth_gbs / 1000 for spec in GPU_SPECS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(names, memory, color='#38bdf8')
axes[0].set_title('Local GPU memory')
axes[0].set_ylabel('GB')
axes[0].tick_params(axis='x', rotation=35)
axes[0].grid(axis='y', alpha=0.25)
axes[1].bar(names, bandwidth, color='#a78bfa')
axes[1].set_title('Peak memory bandwidth')
axes[1].set_ylabel('TB/s')
axes[1].tick_params(axis='x', rotation=35)
axes[1].grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

## 2. Estimate model and KV-cache fit

This is transparent planning arithmetic. Real checkpoints, quantization metadata, allocator behavior, workspaces, and vLLM settings must be measured.

In [ ]:
parameters_b = 7
weight_bytes = 2.0       # FP16/BF16
reserve_gb = 3.0
kv_gb_per_1k_tokens = 0.125
tokens = 4_096

# Experiment 2: calculate weights, KV estimate, reserve, and fit per GPU.
budget = estimate_budget(parameters_b, weight_bytes, reserve_gb, kv_gb_per_1k_tokens, tokens)
print('Budget:', budget)
for spec in GPU_SPECS:
    print(spec.name, 'fits:', budget['total_gb'] <= spec.usable_gb, 'headroom GB:', spec.usable_gb - budget['total_gb'])

In [ ]:
# Compare dtype-dependent weight storage without changing the model architecture.
for label, bytes_per_weight in [('FP16/BF16', 2.0), ('INT8', 1.0), ('INT4', 0.5)]:
    budget = estimate_budget(
        parameters_b, bytes_per_weight, reserve_gb,
        kv_gb_per_1k_tokens, tokens,
    )
    fits = [spec.name for spec in GPU_SPECS if budget['total_gb'] <= spec.usable_gb]
    print(f'{label:10s}: {budget["total_gb"]:6.2f} GB; fits: {fits}')

## 3. Idealized memory-streaming lower bounds

A 14 GB read is a teaching example. It is not a token-generation latency prediction.

In [ ]:
# Experiment 3: idealized one-pass lower bounds for reading 14 GB once.
for spec in GPU_SPECS:
    print(spec.name, 14.0 / spec.bandwidth_gbs * 1000, 'ms')

## 4. Inspect the actual Lightning GPU

Run the next cell on a CUDA runtime. It reports all visible GPUs, free VRAM, compute capability, and SM count.

In [ ]:
# Experiment 4: inspect the actual CUDA device instead of the reference table.
import torch
if not torch.cuda.is_available():
    print('CUDA is unavailable in this environment.')
else:
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        free, total = torch.cuda.mem_get_info(index)
        print(index, props.name, f'{total / 2**30:.2f} GiB total', f'{free / 2**30:.2f} GiB free', f'{props.multi_processor_count} SMs')

In [ ]:
# Optional topology check; this is useful when comparing PCIe and NVLink systems.
import subprocess
subprocess.run(['nvidia-smi', 'topo', '-m'], check=False)

## Interpretation checklist

1. Does the workload fit with runtime headroom?
2. Is it likely bandwidth-sensitive, compute-sensitive, or communication-sensitive?
3. Does the selected dtype have an efficient kernel on the chosen GPU?
4. Would independent replicas avoid communication that tensor parallelism would introduce?
5. What end-to-end throughput, latency, memory, and cost measurements remain to be collected?

A peak specification is a ceiling. The request mix, serving configuration, and actual kernels determine how close the workload gets to it.